In [9]:
import os
import random
import re
import pandas as pd
import tkinter as tk
from tkinter import messagebox
from PIL import Image, ImageTk
from pathlib import Path

from xgboost import train

# === KONFIGURASI GLOBAL (Sesuaikan dengan setup Anda) ===
CSV = "train.csv"         # Jalur ke file CSV Anda
CROPS = Path("./crops")         # Folder tempat menyimpan gambar/crop
SEED = 42                       # Seed acak untuk mengunci urutan data
SKIP = "[SKIP]"                 # Penanda untuk data yang dilewati

def clean_text(text):
    """
    Fungsi pembersih teks (contoh implementasi).
    Sesuaikan isinya dengan aturan pembersihan data Anda.
    """
    if not text:
        return ""
    # Menghapus spasi ganda dan spasi di awal/akhir teks
    cleaned = re.sub(r'\s+', ' ', text).strip()
    return cleaned


In [10]:
def run_label_tool():
    import tkinter as tk
    from tkinter import messagebox
    from PIL import Image, ImageTk

    VIEW_W, VIEW_H = 900, 420
    df = pd.read_csv(CSV, dtype=str, keep_default_na=False)
    order = list(df.index); random.Random(SEED).shuffle(order)      # acak supaya train & val sama-sama terisi
    start = next((p for p, i in enumerate(order) if df.at[i, "text"] == ""), 0)

    root = tk.Tk(); root.title("Label entete")
    state = {"pos": start, "rot": 0, "photo": None}

    canvas = tk.Label(root, bg="#222"); canvas.pack(padx=10, pady=10)
    info = tk.Label(root, font=("Arial", 11)); info.pack()
    preview = tk.Label(root, font=("Consolas", 12), fg="#0a6"); preview.pack()
    text_var = tk.StringVar()
    entry = tk.Entry(root, textvariable=text_var, font=("Consolas", 16), width=60); entry.pack(padx=10, pady=6)
    text_var.trace_add("write", lambda *_: preview.config(text="disimpan sebagai: " + clean_text(text_var.get())))

    def show():
        row = df.loc[order[state["pos"]]]
        im = Image.open(CROPS / row["file"]).convert("RGB").rotate(-state["rot"], expand=True)  # -rot = searah jarum jam
        s = min(VIEW_W / im.width, VIEW_H / im.height)
        im = im.resize((int(im.width * s), int(im.height * s)), Image.LANCZOS)
        state["photo"] = ImageTk.PhotoImage(im); canvas.config(image=state["photo"])
        done = int((df["text"] != "").sum())
        info.config(text=f"[{state['pos'] + 1}/{len(order)}]  {row['file']}  |  rot = {state['rot']}  |  terlabel: {done}")

    def goto(pos):
        state["pos"] = max(0, min(len(order) - 1, pos))
        row = df.loc[order[state["pos"]]]
        state["rot"] = int(row["rot"]) if row["rot"] != "" else 0
        text_var.set("" if row["text"] in ("", SKIP) else row["text"])
        show(); entry.focus_set()

    def rotate(_=None):
        state["rot"] = (state["rot"] + 90) % 360; show()

    def save_and_next(_=None):
        cleaned = clean_text(text_var.get())
        if not cleaned:
            messagebox.showwarning("Teks kosong", "Isi teksnya dulu, atau pakai Lewati."); return
        i = order[state["pos"]]
        df.at[i, "rot"], df.at[i, "text"] = str(state["rot"]), cleaned
        df.to_csv(CSV, index=False); goto(state["pos"] + 1)

    def skip(_=None):
        i = order[state["pos"]]
        df.at[i, "rot"], df.at[i, "text"] = "0", SKIP
        df.to_csv(CSV, index=False); goto(state["pos"] + 1)

    bar = tk.Frame(root); bar.pack(pady=6)
    for label, fn in [("Kembali", lambda: goto(state["pos"] - 1)), ("Putar 90 (Ctrl+R)", rotate),
                      ("Lewati", skip), ("Simpan & Lanjut (Enter)", save_and_next)]:
        tk.Button(bar, text=label, command=fn, width=22).pack(side="left", padx=4)
    root.bind("<Return>", save_and_next); root.bind("<Control-r>", rotate)
    goto(state["pos"]); root.mainloop()

run_label_tool()

KeyError: 'text'

In [ ]:
 # Cek progres labeling
labeled = load_labeled()
print("terlabel:", len(labeled), "| train:", (labeled.split == "train").sum(), "| val:", (labeled.split == "val").sum())
print("rot:", labeled["rot"].value_counts().to_dict())
labeled[["file", "rot", "text"]].head(8)